# Use Case 2: Target-Centric Queries

**Question:** *Given a target protein, which compounds act on it, how potent and selective are they, and are there probes with controls?*

**Example target:** EGFR (Epidermal Growth Factor Receptor) — a key oncology target in lung cancer

This notebook demonstrates how to query the P&D database starting from a **gene name** and traversing to compounds, potency rankings, probes, and scaffolds.

## Data quality filters applied

All activity-based queries in this notebook apply three quality filters by default:

1. **Log-scale only** (`log_only=True`): Restricts to pIC50, pKd, pKi, pEC50, pAC50, pPotency — excludes percentage-scale types (Inhibition 0–100%, Dmax) that cannot be compared on the same axis.
2. **Confidence ≤ 1** (`min_confidence=1`): Keeps only directly measured values, excluding derived/converted values (confidence=2).
3. **Exact values only** (`exact_only=True`): Excludes screening negatives where the only measurement uses the `>` operator (e.g. "IC50 > 30 µM" means the compound was *inactive*, not potent).

Without these filters, EGFR appears to have 2,234 compounds; with them, 1,700 — the difference is percentage-scale and low-confidence measurements that would distort potency rankings.

## Schema paths used

```
basetarget → targettobasetarget → target → activity → compound          (compounds + potency)
basetarget → compoundbasetargetcriteria → compound                       (selectivity)
basetarget → probetobasetarget → probe → compound                        (probes)
probe → probecontrol → compound                                          (controls)
compound → compoundtoscaffold → scaffold → scaffoldtype                  (pre-computed scaffolds)
```

## Setup

In [1]:
import sys
sys.path.insert(0, '/mnt/results')
from pd_utils import *

---
## 2a. Which compounds target EGFR, and what sets do they come from?

The `get_target_compounds()` function returns all compounds with activity against a gene, along with their set memberships. It filters to **log-scale activity types** (pIC50, pKd, pKi, pEC50, pAC50, pPotency) and **confidence ≤ 1** (directly measured values only), excluding percentage-scale measurements (Inhibition, Dmax) and derived values that would inflate the compound count.

In [2]:
# All compounds targeting EGFR
df_egfr_compounds = get_target_compounds('EGFR')
print(f"Total compounds targeting EGFR: {len(df_egfr_compounds)}")
df_egfr_compounds.head(15)

Raw SQL output (1700 rows):
  pdid | compound_name | n_sets | set_types
  --------------------------------------------------------------------------------
  PD003385 | GEFITINIB | 42 | Other non-commercial comp
  PD001319 | IMATINIB | 40 | Other non-commercial comp
  PD001323 | VORINOSTAT | 40 | Other non-commercial comp
  PD003663 | SUNITINIB | 40 | Other non-commercial comp
  PD003446 | DASATINIB | 39 | Other non-commercial comp
  PD002411 | CLOTRIMAZOLE | 38 | Commercial compound sets,
  PD002945 | ZAFIRLUKAST | 38 | Other non-commercial comp
  PD003373 | AFATINIB | 38 | Other non-commercial comp
  PD003493 | SORAFENIB | 38 | Other non-commercial comp
  PD004094 | CERITINIB | 37 | Other non-commercial comp
  PD004100 | PONATINIB | 37 | Other non-commercial comp
  PD004099 | bosutinib | 36 | Other non-commercial comp
  PD003215 | CABOZANTINIB | 35 | Other non-commercial comp
  PD003356 | ERLOTINIB | 35 | Other non-commercial comp
  PD003505 | LAPATINIB | 35 | Other non-commercial com

,pdid,compound_name,n_sets,set_types
0,PD003385,GEFITINIB,42,"Other non-commercial compound sets,Commercial ..."
1,PD001319,IMATINIB,40,"Other non-commercial compound sets,Commercial ..."
2,PD001323,VORINOSTAT,40,"Other non-commercial compound sets,Commercial ..."
3,PD003663,SUNITINIB,40,"Other non-commercial compound sets,Drug compou..."
4,PD003446,DASATINIB,39,"Other non-commercial compound sets,Commercial ..."
5,PD002411,CLOTRIMAZOLE,38,"Commercial compound sets,Other non-commercial ..."
6,PD002945,ZAFIRLUKAST,38,"Other non-commercial compound sets,Drug compou..."
7,PD003373,AFATINIB,38,"Other non-commercial compound sets,Probe compo..."
8,PD003493,SORAFENIB,38,"Other non-commercial compound sets,Drug compou..."
9,PD004094,CERITINIB,37,"Other non-commercial compound sets,Commercial ..."


### Breakdown by set type

The `get_target_settype_breakdown()` function counts compounds per set type (probes, drugs, commercial, etc.), applying the same log-scale and confidence filters as `get_target_compounds()`.

In [3]:
# EGFR compounds by set type
df_egfr_settypes = get_target_settype_breakdown('EGFR')
df_egfr_settypes

Raw SQL output (6 rows):
  set_type | n_compounds
  --------------------------------------------------------------------------------
  Other non-commercial comp | 1395
  Commercial compound sets | 386
  Drug compound sets | 325
  Probe compound sets | 193
   | 57
  Other bioactive compounds | 11



,set_type,n_compounds
0,Other non-commercial compound sets,1395
1,Commercial compound sets,386
2,Drug compound sets,325
3,Probe compound sets,193
4,,57
5,Other bioactive compounds,11


---
## 2b. Which compound is most potent? Which is most selective?

### Most potent

The `get_most_potent_compounds()` function ranks compounds by best potency (MAX of activity values), applying three filters:
- **Log-scale only**: pIC50, pKd, pKi, pEC50, pAC50, pPotency
- **Confidence ≤ 1**: directly measured values only
- **Exact values only** (`value_type = '='`): excludes `>` screening negatives — a compound with only "IC50 > 30 µM" would otherwise appear in the ranking with a misleadingly low pIC50 of ~4.5

In [4]:
# Top 10 most potent EGFR compounds
df_most_potent = get_most_potent_compounds('EGFR', limit=10)
df_most_potent

Raw SQL output (10 rows):
  pdid | compound_name | best_potency | activity_types | n_measurements
  --------------------------------------------------------------------------------
  PD003373 | AFATINIB | 11.0 | pEC50,pIC50,pKd,pKi,pPote | 143
  PD055118 | mobocertinib | 11.0 | pIC50,pKd | 12
  PD003298 | NERATINIB | 10.7 | pEC50,pIC50,pKd,pKi | 51
  PD009249 | OSIMERTINIB | 10.7 | pAC50,pEC50,pIC50,pKd,pKi | 305
  PD125997 | BI-4020 | 10.7 | pIC50 | 5
  PD003483 | PD153035 | 10.6 | pIC50 | 43
  PD015744 | BPIQ-I | 10.6 | pIC50 | 1
  PD003356 | ERLOTINIB | 10.54 | pIC50,pKd,pKi,pPotency | 222
  PD193557 | PD193557 | 10.52 | pIC50,pKi | 5
  PD135743 | ML-03 | 10.43 | pIC50 | 6



,pdid,compound_name,best_potency,activity_types,n_measurements
0,PD003373,AFATINIB,11.00,"pEC50,pIC50,pKd,pKi,pPotency",143
1,PD055118,mobocertinib,11.00,"pIC50,pKd",12
2,PD003298,NERATINIB,10.70,"pEC50,pIC50,pKd,pKi",51
3,PD009249,OSIMERTINIB,10.70,"pAC50,pEC50,pIC50,pKd,pKi",305
4,PD125997,BI-4020,10.70,pIC50,5
5,PD003483,PD153035,10.60,pIC50,43
6,PD015744,BPIQ-I,10.60,pIC50,1
7,PD003356,ERLOTINIB,10.54,"pIC50,pKd,pKi,pPotency",222
8,PD193557,PD193557,10.52,"pIC50,pKi",5
9,PD135743,ML-03,10.43,pIC50,6


### Most selective

The `get_most_selective_compounds()` function uses pre-computed selectivity scores from the curated `compoundbasetargetcriteria` table. The `selectivity` column is the **selectivity window**: the difference (in log units) between the compound's potency against this target and its potency against the next-best off-target. Higher = more selective. These are quality-controlled curated values, not raw MAX(activity_value).

In [5]:
# Top 10 most selective EGFR compounds
df_most_sel = get_most_selective_compounds('EGFR', limit=10)
df_most_sel

Raw SQL output (10 rows):
  pdid | compound_name | potency | selectivity | selectivity_score | family_selectivity | potency_selectivity_synergy
  --------------------------------------------------------------------------------
  PD134676 | PD134676 | 9.52 | 4.3 | 2.0 | NULL | 0
  PD084149 | PD084149 | 8.06 | 4.02 | 2.0 | NULL | 1
  PD017867 | Compound 56 | 11.22 | 3.92 | 2.0 | NULL | 1
  PD083252 | PD083252 | 8.22 | 3.7 | 2.0 | NULL | 1
  PD134674 | PD134674 | 9.3 | 3.41 | 2.0 | NULL | 0
  PD134677 | PD134677 | 8.49 | 3.28 | 2.0 | NULL | 0
  PD083672 | PD083672 | 8.7 | 3.26 | 2.0 | NULL | 1
  PD134673 | PD134673 | 8.28 | 3.21 | 2.0 | NULL | 1
  PD012960 | FALNIDAMOL | 8.52 | 3.05 | 2.0 | NULL | 0
  PD082016 | PD082016 | 8.01 | 3.01 | 2.0 | NULL | 1



,pdid,compound_name,potency,selectivity,selectivity_score,family_selectivity,potency_selectivity_synergy
0,PD134676,PD134676,9.52,4.30,2.0,None,0
1,PD084149,PD084149,8.06,4.02,2.0,None,1
2,PD017867,Compound 56,11.22,3.92,2.0,None,1
3,PD083252,PD083252,8.22,3.70,2.0,None,1
4,PD134674,PD134674,9.30,3.41,2.0,None,0
5,PD134677,PD134677,8.49,3.28,2.0,None,0
6,PD083672,PD083672,8.70,3.26,2.0,None,1
7,PD134673,PD134673,8.28,3.21,2.0,None,1
8,PD012960,FALNIDAMOL,8.52,3.05,2.0,None,0
9,PD082016,PD082016,8.01,3.01,2.0,None,1


### Visualization: Potency vs Selectivity

The `plot_potency_vs_selectivity()` function creates a scatter plot where:
- **X-axis**: Curated potency (−log₁₀ M) from the `compoundbasetargetcriteria` table — this is a quality-controlled value (not raw MAX), so it may be a median or expert-selected measurement rather than the single highest value
- **Y-axis**: Selectivity window (log units) — the gap between potency against EGFR and potency against the next-best off-target; higher = more selective
- **Green points**: compounds flagged with potency-selectivity synergy (both potent AND selective)
- **Grey points**: compounds lacking synergy

The top 5 most selective compounds are labelled. This plot helps identify compounds that are both potent and selective — ideal probe candidates.

In [6]:
# Potency vs selectivity scatter for all EGFR compounds
df_ps = get_potency_selectivity('EGFR')
print(f"Compounds with both potency and selectivity data: {len(df_ps)}")

plot_potency_vs_selectivity(
    df_ps,
    title='EGFR Compounds: Potency vs Selectivity',
    save_path='/mnt/results/notebooks/fig_us2_potency_vs_selectivity.png'
)

Compounds with both potency and selectivity data: 88


---
## 2c. Are there probes with control compounds?

The `get_target_probes()` function uses the **`probetobasetarget`** and **`probecontrol`** tables to find curated probes targeting a gene, along with their negative control compounds.

This is the proper schema path (not the workaround used in the original notebook).

In [7]:
# Probes targeting EGFR with controls
df_egfr_probes = get_target_probes('EGFR')

# Summarize
distinct_probes = df_egfr_probes.drop_duplicates(subset=['probe_pdid'])
with_controls = df_egfr_probes[df_egfr_probes['control_name'].notna()]
print(f"Distinct probes targeting EGFR: {len(distinct_probes)}")
print(f"Probes with named control compounds: {len(with_controls.drop_duplicates(subset=['probe_pdid']))}")
print()

# Show probes that have named controls (not just flag='1')
df_with_named_controls = df_egfr_probes[df_egfr_probes['control_name'].notna()].drop_duplicates(subset=['probe_pdid', 'control_name'])
print(f"Named control entries: {len(df_with_named_controls)}")
df_with_named_controls[['target_gene', 'probe_pdid', 'probe_name', 'control_name', 'probe_origin']].head(20)

Raw SQL output (116 rows):
  target_gene | probe_pdid | probe_name | probe_origin | obsolete_flag | control_flag | control_name | control_compound_id | control_smiles
  --------------------------------------------------------------------------------
  EGFR | PD003373 | AFATINIB | experimental | 1 | 0 | NULL | NULL | NULL
  EGFR | PD003373 | AFATINIB | calculated | 1 | 0 | NULL | NULL | NULL
  EGFR | PD079037 | AG 1478 hydrochloride | calculated | 1 | 1 | 1 | NULL | NULL
  EGFR | PD063235 | AV-412 free base | calculated | 1 | 1 | 1 | NULL | NULL
  EGFR | PD215101 | BI-4732 | experimental | 1 | 0 | NULL | NULL | NULL
  EGFR | PD200883 | BI-8128 | experimental | 1 | 0 | NULL | NULL | NULL
  EGFR | PD003440 | CANERTINIB | experimental | 1 | 1 | NULL | NULL | NULL
  EGFR | PD003440 | CANERTINIB | calculated | 1 | 1 | 1 | NULL | NULL
  EGFR | PD012958 | CL-387785 | calculated | 1 | 1 | 1 | NULL | NULL
  EGFR | PD140853 | DDC-01-163 | experimental | 1 | 0 | NULL | NULL | NULL
  EGFR | PD16428

,target_gene,probe_pdid,probe_name,control_name,probe_origin
2,EGFR,PD079037,AG 1478 hydrochloride,1,calculated
3,EGFR,PD063235,AV-412 free base,1,calculated
7,EGFR,PD003440,CANERTINIB,1,calculated
8,EGFR,PD012958,CL-387785,1,calculated
12,EGFR,PD075489,ICX5600078,1,calculated
13,EGFR,PD075582,ICX5600079,1,calculated
17,EGFR,PD082111,OSI-413,1,calculated
18,EGFR,PD015762,PD 174265,1,calculated
19,EGFR,PD004324,PD004324,1,calculated
23,EGFR,PD080829,PD080829,1,calculated


---
## 2d. Structural similarity via Murcko scaffolds

The `get_murcko_scaffolds()` function retrieves **pre-computed Murcko scaffolds** from the `compoundtoscaffold` + `scaffold` tables — no RDKit needed. It filters to log-scale activity types and confidence ≤ 1, then ranks by best potency.

The **Murcko scaffold** is the core ring system of a molecule with all substituents stripped — compounds sharing a scaffold likely share a binding mode. This is faster and more consistent than computing scaffolds on-the-fly.

In [8]:
# Pre-computed Murcko scaffolds for top 100 most potent EGFR compounds
df_scaffolds = get_murcko_scaffolds('EGFR', limit=100)
print(f"Compound-scaffold rows: {len(df_scaffolds)}")

# Count scaffold frequency
df_scaffold_freq = get_scaffold_frequency(df_scaffolds)
print(f"Unique scaffolds: {len(df_scaffold_freq)}")
print(f"\nTop 5 most common scaffolds:")
df_scaffold_freq.head(5)

Compound-scaffold rows: 100
Unique scaffolds: 57

Top 5 most common scaffolds:


,scaffold_smiles,n_compounds
0,c1ccc(Nc2ncnc3ccccc23)cc1,23
1,c1ccc(Nc2ncnc3cc(OCc4c[nH]nn4)ccc23)cc1,5
2,c1ccc(Nc2ncnc3cc(O[C@H]4CCOC4)ccc23)cc1,4
3,c1ccc(Nc2nccc(-c3c[nH]c4ccccc34)n2)cc1,3
4,c1ccc(Oc2nc(Nc3ccc(N4CCNCC4)cc3)nc3[nH]ccc23)cc1,3


In [9]:
# Visualize top scaffolds
plot_scaffold_frequency(
    df_scaffold_freq,
    title='Top 10 Murcko Scaffolds Among 100 Most Potent EGFR Compounds',
    top_n=10,
    save_path='/mnt/results/notebooks/fig_us2_scaffold_frequency.png'
)

In [10]:
# Show compounds sharing the most common scaffold
top_scaffold = df_scaffold_freq.iloc[0]['scaffold_smiles']
df_top_scaffold = df_scaffolds[df_scaffolds['scaffold_smiles'] == top_scaffold][
    ['pdid', 'compound_name', 'best_potency']
].sort_values('best_potency', ascending=False)
print(f"Compounds sharing the most common scaffold ({df_scaffold_freq.iloc[0]['n_compounds']} compounds):")
df_top_scaffold

Compounds sharing the most common scaffold (23 compounds):


,pdid,compound_name,best_potency
0,PD017867,Compound 56,11.22
7,PD003483,PD153035,10.60
9,PD003356,ERLOTINIB,10.54
11,PD135743,ML-03,10.43
19,PD005576,PD-168393,10.10
26,PD015762,PD 174265,10.00
28,PD187051,PD187051,10.00
30,PD134786,PD134786,9.96
32,PD135745,PD135745,9.92
33,PD187049,PD187049,9.92


---
## Summary

| Metric | Value |
|--------|-------|
| Total EGFR compounds (quality-filtered) | 1,700 |
| Most potent | Afatinib / mobocertinib (pIC50 = 11.0) |
| Most selective | PD134676 (selectivity window = 4.3 log units) |
| Distinct probes | 114 |
| Probes with named controls | 58 |
| Unique Murcko scaffolds (top 100) | 33 |

**Key insight:** EGFR has a rich compound landscape spanning approved drugs (gefitinib, erlotinib, afatinib) and research probes. The pre-computed scaffold table reveals quinazoline as the dominant scaffold among potent EGFR inhibitors. Quality filtering (log-scale, confidence=1, exact values) reduces the compound count from 2,234 to 1,700 by excluding percentage-scale and low-confidence measurements.

## Reusable functions used

| Function | Purpose |
|----------|---------|
| `get_target_compounds(gene)` | All compounds targeting a gene (log-scale, conf≤1) |
| `get_target_settype_breakdown(gene)` | Compounds by set type (same filters) |
| `get_most_potent_compounds(gene, limit)` | Potency ranking (log-scale, conf≤1, exact only) |
| `get_most_selective_compounds(gene, limit)` | Selectivity ranking (curated cbtc table) |
| `get_potency_selectivity(gene)` | Potency + selectivity pairs (curated cbtc table) |
| `get_target_probes(gene)` | Probes with controls (probetobasetarget) |
| `get_murcko_scaffolds(gene, limit)` | Pre-computed scaffolds (log-scale, conf≤1) |
| `get_scaffold_frequency(df)` | Scaffold frequency counting |
| `plot_potency_vs_selectivity(df, ...)` | Scatter: curated potency vs selectivity window |
| `plot_scaffold_frequency(df, ...)` | Scaffold bar chart |